<a href="https://colab.research.google.com/github/prabhakar1234pr/MoviebuffS-Cinema-Insights-App-/blob/main/sentiment_analysis_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle

In [4]:
pip show tensorflow


Name: tensorflow
Version: 2.17.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.10/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: dopamine_rl, tf_keras


In [6]:
pip install --upgrade tensorflow


In [ ]:
import tensorflow as tf
print(tf.__version__)  # Should show the latest TensorFlow version
print(tf.keras.__version__)  # Should match the updated version


In [ ]:
import os  # Standard library for operating system functionality
import json  # Standard library for JSON handling
from zipfile import ZipFile  # Standard library for ZIP file handling

import pandas as pd  # For data manipulation and analysis
from sklearn.model_selection import train_test_split  # For splitting the dataset

import tensorflow as tf  # Ensure you're using TensorFlow correctly
from tensorflow.keras.models import Sequential  # For building a sequential model
from tensorflow.keras.layers import Dense, Embedding, LSTM  # LSTM layers for the model
from tensorflow.keras.preprocessing.text import Tokenizer  # For text tokenization
from tensorflow.keras.preprocessing.sequence import pad_sequences  # For padding sequences

# Optional: Suppress warnings (not necessary)
import warnings
warnings.filterwarnings('ignore')


### **Data Collection From Kaggle API**

In [ ]:
kaggle_dictionary = json.load(open("kaggle.json"))

In [ ]:
#set up kaggle credentials as environment variables
os.environ["KAGGLE_USERNAME"] = kaggle_dictionary['username']
os.environ["KAGGLE_KEY"] = kaggle_dictionary['key']


In [ ]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [ ]:
!ls

In [ ]:
!unzip imdb-dataset-of-50k-movie-reviews.zip


Archive:  imdb-dataset-of-50k-movie-reviews.zip
replace IMDB Dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd
df = pd.read_csv('IMDB Dataset.csv')
print(df.head())


In [ ]:
df.tail()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
df.replace({"sentiment":{"positive":1,"negative":0}},inplace = True)

In [ ]:
df.head()

In [ ]:
df['sentiment'].value_counts()

### **Splitting the Data into Training and Testing Data**

In [ ]:
train_data,test_data = train_test_split(df,test_size=0.2,random_state=42)

In [ ]:
print(train_data.shape)

print(test_data.shape)

### **DATA** **PREPROCESSING**

In [ ]:
#Tokenize the data
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['review'])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']),maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']),maxlen=200)

In [ ]:
print(X_test)

In [ ]:
print(X_train)

In [ ]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [ ]:
print( Y_train )

### Building LSTM model

In [ ]:
#create a model
model = Sequential()
model.add(Embedding(input_dim=5000,output_dim=128,input_length = 200 ,input_shape=(200,)))
model.add(LSTM(128,dropout=0.2,recurrent_dropout=0.2))
model.add(Dense(1,activation="sigmoid"))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()


### TRAINING THE MODEL




In [ ]:
model.fit(X_train,Y_train,batch_size=65,epochs=5,validation_split=0.2)

### MODEL EVALUATION

In [ ]:
loss,accuracy = model.evaluate(X_test,Y_test)

In [ ]:
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

In [ ]:
def predict_sentiment(review):
    # Tokenize and pad the review
    review_sequence = tokenizer.texts_to_sequences([review])
    pad_review_sequence = pad_sequences(review_sequence, maxlen=200)

    # Make a prediction
    prediction = model.predict(pad_review_sequence)

    # Interpret the prediction
    if prediction > 0.5:
        return "Positive"
    else:
        return "Negative"

In [ ]:
new_review = "I slept while watching this film"
predicted_sentiment = predict_sentiment(new_review)
print(f"Predicted Sentiment: {predicted_sentiment}")

In [ ]:
!pip install streamlit

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
import streamlit as st


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("2naLl1G29AyWyy75YaiojOUPS7R_5U2LQrN7Wf2TScvLEaaV2")

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
model.save('my_movie_review_model.keras')

In [ ]:
!streamlit run sentiment_analysis_IMDB.py & npx ngrok http 8501